# 제5장(b): 산업별 AI 적용

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch05b.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**Financial AI Fairness Evaluation**

In [ ]:
class FinancialAIFairnessEvaluator:
    """Evaluates 4/5 rule and Adverse Impact for Financial AI."""
    def __init__(self, protected_attrs=['gender', 'age', 'region']):
        self.protected_attrs = protected_attrs

    def evaluate_4_5_rule(self, y_pred, sensitive_features):
        results = {}
        for attr in self.protected_attrs:
            groups = sensitive_features[attr].unique()
            rates = {g: y_pred[sensitive_features[attr]==g].mean() for g in groups}
            max_r = max(rates.values())
            # Adverse Impact Ratio (AIR)
            results[attr] = {g: r/max_r for g, r in rates.items()}
        return results

    def generate_adverse_action_notice(self, shap_vals, feat_desc):
        # Sort factors by negative impact
        sorted_neg = sorted(shap_vals.items(), key=lambda x: x[1])[:4]
        notice = "Credit Decision Notice\n\nKey Factors:\n"
        for i, (f, v) in enumerate(sorted_neg, 1):
            notice += f"{i}. {feat_desc.get(f, f)}\n"
        return notice

**Healthcare AI Validation Framework**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import numpy as np
from sklearn.metrics import (
 roc_auc_score, precision_recall_curve, 
 confusion_matrix, accuracy_score
)

@dataclass
class ClinicalValidationConfig:
 """Healthcare AI Clinical Setup"""
 
 # Analysis Target
 subgroups: List[str] = field(default_factory=lambda: [
 'sex', # 
 'age_group', # 
 'ethnicity', # / 
 'comorbidity', # 'device_type', # ( AI)
 'institution', # Institution ( Institution )
 ])
 
 # Performance 
 min_sensitivity: float = 0.90 # Minimum 
 min_specificity: float = 0.80 # Minimum 
 max_subgroup_gap: float = 0.10 # Performance 
 
 # Interval
 confidence_level: float = 0.95


class MedicalAIValidator:
 """
 Healthcare AI Tool
 
 Guide , FDA GMLP Compliance Clinical Assessment 
 """
 
 def __init__(self, config: Optional[ClinicalValidationConfig] = None):
 self.config = config or ClinicalValidationConfig()
 
 def evaluate_diagnostic_performance(
 self,
 y_true: np.ndarray,
 y_prob: np.ndarray,
 operating_point: Optional[float] = None,
 ) -> Dict:
 """
 AI Performance Assessment
 
 , , PPV, NPV 95% Interval Calculation
 """
 
 # AUC
 auc = roc_auc_score(y_true, y_prob)
 
 # operating point (Youden's J)
 if operating_point is None:
 precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
 # Sensitivity-Specificity based fpr_tpr_thresholds = self._calculate_roc_points(y_true, y_prob)
 operating_point = self._find_optimal_threshold(fpr_tpr_thresholds)
 
 # y_pred = (y_prob >= operating_point).astype(int)
 tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
 
 # Major 
 sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
 specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
 ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
 npv = tn / (tn + fn) if (tn + fn) > 0 else 0
 
 # 95% Interval (Wilson score interval)
 sensitivity_ci = self._wilson_ci(tp, tp + fn)
 specificity_ci = self._wilson_ci(tn, tn + fp)
 ppv_ci = self._wilson_ci(tp, tp + fp)
 npv_ci = self._wilson_ci(tn, tn + fn)
 
 return {
 'auc': auc,
 'operating_point': operating_point,
 'sensitivity': {'value': sensitivity, 'ci_95': sensitivity_ci},
 'specificity': {'value': specificity, 'ci_95': specificity_ci},
 'ppv': {'value': ppv, 'ci_95': ppv_ci},
 'npv': {'value': npv, 'ci_95': npv_ci},
 'confusion_matrix': {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp},
 'meets_sensitivity_threshold': sensitivity >= self.config.min_sensitivity,
 'meets_specificity_threshold': specificity >= self.config.min_specificity,
 }
 
 def evaluate_subgroup_performance(
 self,
 y_true: np.ndarray,
 y_prob: np.ndarray,
 subgroup_data: Dict[str, np.ndarray],
 operating_point: float,
 ) -> Dict:
 """
 Performance Analysis
 
 Equity Assessment: , , Performance Analysis
 """
 
 results = {}
 
 for subgroup_name, subgroup_labels in subgroup_data.items():
 unique_groups = np.unique(subgroup_labels)
 group_performance = {}
 
 for group in unique_groups:
 mask = subgroup_labels == group
 if mask.sum() < 30: # Minimum Number of samples
 continue
 
 group_y_true = y_true[mask]
 group_y_prob = y_prob[mask]
 
 perf = self.evaluate_diagnostic_performance(
 group_y_true, group_y_prob, operating_point
 )
 
 group_performance[str(group)] = {
 'n': mask.sum(),
 'sensitivity': perf['sensitivity']['value'],
 'specificity': perf['specificity']['value'],
 'auc': perf['auc'],
 }
 
 # Performance Analysis
 if len(group_performance) >= 2:
 sensitivities = [g['sensitivity'] for g in group_performance.values()]
 specificities = [g['specificity'] for g in group_performance.values()]
 
 sens_gap = max(sensitivities) - min(sensitivities)
 spec_gap = max(specificities) - min(specificities)
 
 results[subgroup_name] = {
 'groups': group_performance,
 'sensitivity_gap': sens_gap,
 'specificity_gap': spec_gap,
 'equity_concern': sens_gap > self.config.max_subgroup_gap or spec_gap > self.config.max_subgroup_gap,
 }
 
 return results
 
 def _wilson_ci(
 self,
 successes: int,
 trials: int,
 z: float = 1.96,
 ) -> Tuple[float, float]:
 """Wilson score confidence interval"""
 
 if trials == 0:
 return (0.0, 0.0)
 
 z2 = z * z
 trials2 = trials * trials
 p = successes / trials
 denominator = 1 + z2 / trials
 center = (p + z2 / (2 * trials)) / denominator
 margin = z * np.sqrt(p * (1 - p) / trials + z**2 / (4 * trials2)) / denominator
 
 return (max(0, center - margin), min(1, center + margin))
 
 def _calculate_roc_points(
 self,
 y_true: np.ndarray,
 y_prob: np.ndarray,
 ) -> List[Tuple[float, float, float]]:
 """ROC Calculation"""
 
 thresholds = np.unique(y_prob)
 points = []
 
 for thresh in thresholds:
 y_pred = (y_prob >= thresh).astype(int)
 tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
 
 tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
 fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
 
 points.append((fpr, tpr, thresh))
 
 return points
 
 def _find_optimal_threshold(
 self,
 roc_points: List[Tuple[float, float, float]],
 ) -> float:
 """Youden's J statistic based """
 
 best_j = -1
 best_thresh = 0.5
 
 for fpr, tpr, thresh in roc_points:
 j = tpr - fpr # Youden's J
 if j > best_j:
 best_j = j
 best_thresh = thresh
 
 return best_thresh
 
 def generate_clinical_validation_report(
 self,
 model_name: str,
 indication: str,
 performance: Dict,
 subgroup_analysis: Dict,
 ) -> str:
 """ Clinical Report Generation"""
 
 report = f"""
# Healthcare AI Clinical ReportModel : {model_name} : {indication} : Data AI based Healthcare Guide 

---

## 1. Performance

| | | 95% CI | |
|--------|-----|--------|----------|
| AUC | {performance['auc']:.4f} | - | - |
| | {performance['sensitivity']['value']:.4f} | [{performance['sensitivity']['ci_95'][0]:.4f}, {performance['sensitivity']['ci_95'][1]:.4f}] | {'[PASS]' if performance['meets_sensitivity_threshold'] else '[FAIL]'} (>={self.config.min_sensitivity}) |
| | {performance['specificity']['value']:.4f} | [{performance['specificity']['ci_95'][0]:.4f}, {performance['specificity']['ci_95'][1]:.4f}] | {'[PASS]' if performance['meets_specificity_threshold'] else '[FAIL]'} (>={self.config.min_specificity}) |
| (PPV) | {performance['ppv']['value']:.4f} | [{performance['ppv']['ci_95'][0]:.4f}, {performance['ppv']['ci_95'][1]:.4f}] | - |
| (NPV) | {performance['npv']['value']:.4f} | [{performance['npv']['ci_95'][0]:.4f}, {performance['npv']['ci_95'][1]:.4f}] | - |

### | | | |
|--|----------|----------|
| | {performance['confusion_matrix']['tp']} (TP) | {performance['confusion_matrix']['fn']} (FN) |
| | {performance['confusion_matrix']['fp']} (FP) | {performance['confusion_matrix']['tn']} (TN) |

---

## 2. Analysis ( Equity)

"""
 
 equity_concerns = []
 
 for subgroup, result in subgroup_analysis.items():
 report += f"### {subgroup}\n\n"
 
 if result.get('equity_concern', False):
 equity_concerns.append(subgroup)
 report += f"[WARN]Equity : Performance ({self.config.max_subgroup_gap:.0%}) \n\n"
 
 report += "| | N | | | AUC |\n"
 report += "|------|---|--------|--------|-----|\n"
 
 for group, stats in result.get('groups', {}).items():
 report += f"| {group} | {stats['n']} | {stats['sensitivity']:.4f} | {stats['specificity']:.4f} | {stats['auc']:.4f} |\n"
 
 report += f"\n- : {result['sensitivity_gap']:.4f}\n"
 report += f"- : {result['specificity_gap']:.4f}\n\n"
 
 report += """
---

## 3. Assessment """
 
 if not performance['meets_sensitivity_threshold']:
 report += "[FAIL] : Impact \n\n"
 
 if not performance['meets_specificity_threshold']:
 report += "[WARN] : Addition \n\n"
 
 if equity_concerns:
 report += f"[WARN] Equity : {', '.join(equity_concerns)} Performance \n\n"
 
 if (performance['meets_sensitivity_threshold'] and 
 performance['meets_specificity_threshold'] and 
 not equity_concerns):
 report += "[PASS] Clinical **\n\n"
 
 report += """
---

## 4. 1. AI Healthcare Tool , Final Healthcare .
2. Learning Data , Performance .
3. Clinical AI days , Healthcare Decision .

---

* Report 15 5years .*
"""
 
 return report

**Public AI Transparency Report Generator**

In [ ]:
import json
from datetime import datetime
from typing import Dict, List, Any

class PublicAITransparencyReport:
 """
 PublicInstitution AI System Report Generator
 
 EO 14110 PublicInstitution AI Guide Compliance Support
 """
 
 def __init__(self, system_name: str, department: str):
 self.system_info = {
 "system_name": system_name,
 "department": department,
 "generation_date": datetime.now().isoformat(),
 "data_sources": [],
 "algorithms": [],
 "impact_assessment": {},
 "human_oversight": ""
 }

 def add_data_source(self, name: str, source: str, collection_method: str):
 self.system_info["data_sources"].append({
 "name": name,
 "source": source,
 "method": collection_method
 })

 def set_algorithm_details(self, model_type: str, framework: str, version: str):
 self.system_info["algorithms"].append({
 "type": model_type,
 "framework": framework,
 "version": version
 })

 def set_impact_assessment(self, target_population: str, potential_risks: List[str], mitigations: List[str]):
 self.system_info["impact_assessment"] = {
 "target": target_population,
 "risks": potential_risks,
 "mitigations": mitigations
 }

 def set_human_oversight(self, role: str, process: str):
 self.system_info["human_oversight"] = f" : {role}, : {process}"

 def generate_report(self) -> str:
 report = f"# AI System Report: {self.system_info['system_name']}\n\n"
 report += f" : {self.system_info['department']}\n"
 report += f"days: {self.system_info['generation_date']}\n\n"
 
 report += "## 1. Data Usage \n"
 for ds in self.system_info["data_sources"]:
 report += f"- {ds['name']} ({ds['source']}): {ds['method']}\n"
 
 report += "\n## 2. information\n"
 for alg in self.system_info["algorithms"]:
 report += f"- {alg['type']} (ver {alg['version']}), Framework: {alg['framework']}\n"
 
 report += "\n## 3. Impact Assessment \n"
 impact = self.system_info["impact_assessment"]
 report += f"- Target : {impact.get('target', 'N/A')}\n"
 report += "- Major Risk:\n"
 for risk in impact.get('risks', []):
 report += f" - {risk}\n"
 report += "- Response :\n"
 for mit in impact.get('mitigations', []):
 report += f" - {mit}\n"
 
 report += f"\n## 4. \n{self.system_info['human_oversight']}\n"
 
 return report

# Example
reporter = PublicAITransparencyReport(" AI", " Support ")
reporter.add_data_source(" / information", " / ", " Integration")
reporter.set_algorithm_details("Random Forest Regressor", "Scikit-learn", "1.2.0")
reporter.set_impact_assessment(
 " ",
 [" ", " "],
 [" Weights ", " "]
)
reporter.set_human_oversight(" ", "AI list based Final Target ")

print(reporter.generate_report())

**Manufacturing AI Safety Assessment Tool**

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple
import time

@dataclass
class ManufacturingAISafetyConfig:
 """Manufacturing AI Setup"""
 max_latency_ms: float = 50.0 # safety_buffer: float = 0.2 # (20%)
 reliability_threshold: float = 0.999 # emergency_stop_signals: List[str] = None

class ManufacturingAISafetyEvaluator:
 """
 Manufacturing AI Assessment 
 
 ISO 10218 EU AI Act Compliance Support
 """
 
 def __init__(self, config: ManufacturingAISafetyConfig):
 self.config = config
 self.safety_logs = []

 def check_operational_envelope(self, predicted_value: float, limits: Tuple[float, float]) -> bool:
 """
 AI Operation (Operational Envelope) """
 lower_limit, upper_limit = limits
 # Calculation
 safe_lower = lower_limit * (1 + self.config.safety_buffer) if lower_limit > 0 else lower_limit * (1 - self.config.safety_buffer)
 safe_upper = upper_limit * (1 - self.config.safety_buffer)
 
 is_safe = safe_lower <= predicted_value <= safe_upper
 
 if not is_safe:
 self._trigger_safety_alert(f" : {predicted_value} Scope ")
 
 return is_safe

 def measure_realtime_performance(self, start_time: float, end_time: float) -> bool:
 """
 AI """
 latency = (end_time - start_time) * 1000 # ms
 if latency > self.config.max_latency_ms:
 self._trigger_safety_alert(f" : {latency:.2f}ms ( : {self.config.max_latency_ms}ms)")
 return False
 return True

 def _trigger_safety_alert(self, message: str):
 log_entry = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [SAFETY ALERT] {message}"
 self.safety_logs.append(log_entry)
 print(log_entry)
 # PLC(Logical Controller) # Example
config = ManufacturingAISafetyConfig(max_latency_ms=30.0, safety_buffer=0.15)
evaluator = ManufacturingAISafetyEvaluator(config)

# inference_start = time.time()
# ... AI Model ...
time.sleep(0.02) # 20ms inference_end = time.time()

# 1. is_realtime = evaluator.measure_realtime_performance(inference_start, inference_end)

# 2. Scope ( : )
torque_prediction = 85.5
operating_limit = (0.0, 100.0)
is_within_envelope = evaluator.check_operational_envelope(torque_prediction, operating_limit)

if not is_within_envelope or not is_realtime:
 print("System .")

**Industry-Specific AI Governance Configurator**

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
from enum import Enum

class IndustryType(Enum):
    FINANCE = "finance"
    HEALTHCARE = "healthcare"
    PUBLIC_SECTOR = "public_sector"
    MANUFACTURING = "manufacturing"

class RiskLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

@dataclass
class GovernancePolicy:
    """Governance Policy for a Specific Industry and Use Case"""
    industry: IndustryType
    use_case: str
    risk_level: RiskLevel
    required_approvals: List[str]
    monitoring_frequency: str
    explainability_requirements: List[str]
    human_oversight_type: str
    data_protection_measures: List[str]
    audit_requirements: List[str]
    genai_specific_controls: List[str]

class IndustryGovernanceConfigurator:
    """
    Configures governance policy based on industry type.

    Generates tailored governance requirements considering
    industry regulations, risk profiles, and GenAI considerations.
    """

    # Industry-specific governance templates
    INDUSTRY_PROFILES = {
        IndustryType.FINANCE: {
            "primary_values": ["fairness", "explainability", "consumer_protection"],
            "key_regulations": ["FSC_AI_Guideline", "SR_11-7", "ECOA", "AI_Basic_Act"],
            "default_monitoring": "quarterly",
            "human_oversight": "model_validation_committee",
            "genai_controls": [
                "Hallucination rate monitoring (<2%)",
                "Deepfake detection for KYC/AML",
                "Prompt filtering for PII",
                "RAG-based response verification",
                "AI agent action audit trail",
                "Pre-approval for all GenAI use cases",
            ],
        },
        IndustryType.HEALTHCARE: {
            "primary_values": ["patient_safety", "clinical_validity", "health_equity"],
            "key_regulations": ["MFDS_Guideline", "Medical_Device_Act", "FDA_GMLP"],
            "default_monitoring": "continuous",
            "human_oversight": "clinician_final_decision",
            "genai_controls": [
                "Clinical information accuracy validation",
                "Patient consent for AI-generated advice",
                "Prohibition of unsupervised diagnosis",
                "Medical terminology hallucination check",
                "Subgroup performance equity monitoring",
            ],
        },
        IndustryType.PUBLIC_SECTOR: {
            "primary_values": ["democratic_values", "due_process", "transparency"],
            "key_regulations": ["Intelligence_Info_Act", "Admin_Procedure_Act", "PIPA"],
            "default_monitoring": "annual_report",
            "human_oversight": "official_final_approval",
            "genai_controls": [
                "AI-generated document labeling",
                "Citizen notification of AI use",
                "Sensitive admin data input prohibition",
                "Chatbot accuracy threshold (>98%)",
                "Accessibility for vulnerable populations",
                "Version control for AI-drafted documents",
            ],
        },
        IndustryType.MANUFACTURING: {
            "primary_values": ["physical_safety", "reliability", "real_time"],
            "key_regulations": ["ISO_10218", "SDPA", "IEC_62443"],
            "default_monitoring": "real_time",
            "human_oversight": "supervisor_approval_with_e_stop",
            "genai_controls": [
                "Process parameter generation safety bounds",
                "Operational envelope validation",
                "Fail-safe mode for GenAI suggestions",
                "Latency constraints for real-time control",
            ],
        },
    }

    RISK_ESCALATION = {
        RiskLevel.LOW: {
            "approvals": ["team_lead"],
            "audit_cycle": "annual",
            "explainability": ["global_feature_importance"],
        },
        RiskLevel.MEDIUM: {
            "approvals": ["team_lead", "governance_officer"],
            "audit_cycle": "semi_annual",
            "explainability": ["global_feature_importance", "local_explanations"],
        },
        RiskLevel.HIGH: {
            "approvals": ["team_lead", "governance_officer", "legal_review"],
            "audit_cycle": "quarterly",
            "explainability": [
                "global_feature_importance", "local_explanations",
                "counterfactual_analysis", "adverse_action_notice"
            ],
        },
        RiskLevel.CRITICAL: {
            "approvals": [
                "team_lead", "governance_officer",
                "legal_review", "board_approval"
            ],
            "audit_cycle": "monthly",
            "explainability": [
                "global_feature_importance", "local_explanations",
                "counterfactual_analysis", "adverse_action_notice",
                "full_model_documentation"
            ],
        },
    }

    def configure(
        self,
        industry: IndustryType,
        use_case: str,
        risk_level: RiskLevel,
        uses_genai: bool = False,
    ) -> GovernancePolicy:
        """Generate governance policy for given industry and use case."""

        profile = self.INDUSTRY_PROFILES[industry]
        risk_config = self.RISK_ESCALATION[risk_level]

        # Data protection based on industry
        data_measures = self._get_data_protection(industry)

        # GenAI-specific controls
        genai_controls = profile["genai_controls"] if uses_genai else []

        # Monitoring: use stricter of risk-based and industry default
        monitoring = self._resolve_monitoring(
            risk_config["audit_cycle"], profile["default_monitoring"]
        )

        return GovernancePolicy(
            industry=industry,
            use_case=use_case,
            risk_level=risk_level,
            required_approvals=risk_config["approvals"],
            monitoring_frequency=monitoring,
            explainability_requirements=risk_config["explainability"],
            human_oversight_type=profile["human_oversight"],
            data_protection_measures=data_measures,
            audit_requirements=[
                f"Audit cycle: {risk_config['audit_cycle']}",
                f"Key regulations: {', '.join(profile['key_regulations'])}",
                "Independent validation required"
                    if risk_level in (RiskLevel.HIGH, RiskLevel.CRITICAL)
                    else "Self-assessment acceptable",
            ],
            genai_specific_controls=genai_controls,
        )

    def _get_data_protection(self, industry: IndustryType) -> List[str]:
        measures = {
            IndustryType.FINANCE: [
                "Credit information encryption",
                "Transaction data anonymization",
                "5-year record retention",
            ],
            IndustryType.HEALTHCARE: [
                "PHI de-identification (Safe Harbor)",
                "IRB approval for research use",
                "Pseudonymization per PIPA",
            ],
            IndustryType.PUBLIC_SECTOR: [
                "Administrative data anonymization",
                "Citizen consent management",
                "Data minimization principle",
            ],
            IndustryType.MANUFACTURING: [
                "OT/IT network segmentation",
                "Trade secret protection",
                "Sensor data integrity verification",
            ],
        }
        return measures.get(industry, [])

    def _resolve_monitoring(self, risk_freq: str, industry_freq: str) -> str:
        priority = ["real_time", "monthly", "quarterly", "semi_annual", "annual", "annual_report"]
        risk_idx = priority.index(risk_freq) if risk_freq in priority else 5
        ind_idx = priority.index(industry_freq) if industry_freq in priority else 5
        return priority[min(risk_idx, ind_idx)]

    def generate_summary(self, policy: GovernancePolicy) -> str:
        """Generate human-readable governance policy summary."""
        summary = f"""
=== AI Governance Policy Summary ===
Industry: {policy.industry.value}
Use Case: {policy.use_case}
Risk Level: {policy.risk_level.value}

[Approvals Required]
{chr(10).join(f'  - {a}' for a in policy.required_approvals)}

[Monitoring]: {policy.monitoring_frequency}
[Human Oversight]: {policy.human_oversight_type}

[Explainability Requirements]
{chr(10).join(f'  - {e}' for e in policy.explainability_requirements)}

[Data Protection]
{chr(10).join(f'  - {d}' for d in policy.data_protection_measures)}

[Audit Requirements]
{chr(10).join(f'  - {a}' for a in policy.audit_requirements)}
"""
        if policy.genai_specific_controls:
            summary += f"""
[GenAI-Specific Controls]
{chr(10).join(f'  - {g}' for g in policy.genai_specific_controls)}
"""
        return summary


# Usage Example
configurator = IndustryGovernanceConfigurator()

# Finance: LLM-based customer chatbot (high risk, uses GenAI)
finance_policy = configurator.configure(
    industry=IndustryType.FINANCE,
    use_case="LLM-based customer service chatbot",
    risk_level=RiskLevel.HIGH,
    uses_genai=True,
)
print(configurator.generate_summary(finance_policy))

# Public Sector: AI-assisted document drafting (medium risk, uses GenAI)
public_policy = configurator.configure(
    industry=IndustryType.PUBLIC_SECTOR,
    use_case="AI-assisted administrative document drafting",
    risk_level=RiskLevel.MEDIUM,
    uses_genai=True,
)
print(configurator.generate_summary(public_policy))